# CJD-Spatial-Dynamics: Mathematical Proofs and Foundations
### Documenting the Physics-Based ML and Stochastic Logic

This notebook serves as the academic validation for the **CJD-Spatial-Dynamics** engine. It proves the first principles behind our tracking algorithms, focusing on outlier resistance and signal isolation in high-entropy environments.

## 1. Modeling Discrete Event Probability (Poisson Theory)
At RADAR, tracking a person passing an RFID gate or a product being moved is a discrete event. We model this using a **Poisson Process**.

**The Axiom:** The intensity ($\lambda$) represents the average rate of "events" (collisions or detections) per spatial sector.
$$P(k \text{ detections in interval}) = \frac{\lambda^k e^{-\lambda}}{k!}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import poisson

# Visualize the effect of increasing Lambda (detection intensity) 
# on the probability of 'missing' a tag (k=0)
lambdas = [0.5, 1.0, 2.0, 5.0]
k_vals = np.arange(0, 15)

plt.figure(figsize=(10, 4))
for lam in lambdas:
    # Using 'lam' instead of 'l' for PEP 8 visual clarity
    plt.plot(k_vals, poisson.pmf(k_vals, lam), '-o', label=f'Intensity (λ) = {lam}')

plt.title("Poisson Distribution: Detection Probability per Spatial Sector")
plt.xlabel("Number of Detections (k)")
plt.ylabel("Probability")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 2. Chiral Momentum (z): Predicting Path Correction
Physical assets often exhibit **Mean Reversion**. If a customer walks too far from an aisle center, they are statistically likely to correct their path. We measure this "Tension" using a rolling Z-score.

**The Formula:**
$$z = \frac{x_t - \mu_{rolling}}{\sigma_{rolling}}$$ 
A high $|z|$ indicates "Path Elasticity," triggering a weighted adjustment in the predictive model.

In [ ]:
import pandas as pd

# Simulating a walk with a sudden deviation
np.random.seed(42)
path = np.cumsum(np.random.normal(0, 1, 100))
path[80:] += 10 # Sudden deviation (the 'tension' event)

df = pd.DataFrame({'pos': path})
df['mu'] = df['pos'].rolling(window=10).mean()
df['sigma'] = df['pos'].rolling(window=10).std()
df['z_score'] = (df['pos'] - df['mu']) / df['sigma']

# Visualization of "Tension"
fig, ax1 = plt.subplots(figsize=(10, 4))
ax1.plot(df['pos'], color='cyan', label='Asset Position')
ax2 = ax1.twinx()
ax2.fill_between(df.index, df['z_score'], 0, color='red', alpha=0.3, label='Chiral Tension (z)')
plt.title("Path Elasticity: Detecting Off-Trajectory Events")
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')
plt.show()

## 3. Medoid Convergence: Isolating the 'True' Coordinate
Standard K-Means is sensitive to outliers. In a store with "multi-path" RFID interference, we use **K-Medoids**. This ensures the centroid is an *actual observed point*, not a skewed average.

In [ ]:
# Note: In a live environment, pip install scikit-learn-extra
from sklearn.cluster import KMeans

# Generate a noisy sensor cloud with 10% 'Ghost' outliers
data = np.random.normal(0, 1, (100, 1))
outliers = np.random.normal(20, 1, (10, 1))
cloud = np.vstack([data, outliers])

mean_val = np.mean(cloud)
# Simulating Medoid behavior by using a median-based robust metric
medoid_val = np.median(cloud) 

plt.figure(figsize=(10,4))
plt.hist(cloud, bins=30, alpha=0.5, color='gray', label='Sensor Readings')
plt.axvline(mean_val, color='red', linestyle='--', label='Mean (Skews toward Outliers)')
plt.axvline(medoid_val, color='green', linewidth=3, label='Robust Centroid (Stable)')
plt.title("Outlier Resilience: Why Medoid Logic Wins in High-Noise Environments")
plt.legend()
plt.show()

## 4. Entropy-Adjusted Predictions
We use **Shannon Entropy** to measure the "Chaos" of a data stream. If entropy is high, we apply a **Beta Adjustment ($\beta$)** to penalize model confidence.

**Entropy Formula:**
$$H(X) = -\sum P(x_i) \log_2 P(x_i)$$
The Uncertainty Penalty ensures automation systems do not trigger on low-confidence data.

In [ ]:
import math

def calculate_entropy(probs):
    return -sum(p * math.log2(p) for p in probs if p > 0)

# Scenario A: High Confidence (Tag is definitely on Shelf 1)
probs_a = [0.95, 0.05] 
# Scenario B: High Chaos (Tag could be anywhere)
probs_b = [0.5, 0.5]

print(f"Entropy A (Signal): {calculate_entropy(probs_a):.4f} bits")
print(f"Entropy B (Noise):  {calculate_entropy(probs_b):.4f} bits")